# Notebook 02 – Unsupervised Learning: Kundensegmentierung

Dieses Notebook baut direkt auf dem bereinigten Datensatz aus Notebook 01 auf.
Ziel ist es, Kund:innen anhand ihres Kaufverhaltens automatisch in sinnvolle Gruppen einzuteilen –
ohne vorgegebene Zielvariable.

**Ablauf:**
1. Bereinigten Datensatz laden
2. Kundenbezogene Merkmale berechnen
3. Log-Transformation und Skalierung
4. Optimale Clusteranzahl bestimmen (Elbow + Silhouette)
5. K-Means Clustering durchführen
6. Cluster interpretieren und visualisieren (PCA)
7. Hierarchisches Clustering als Vergleich
8. Ergebnisse exportieren

## 1. Bibliotheken importieren

In [ ]:
# Datenverarbeitung
import pandas as pd
import numpy as np

# Visualisierung
import matplotlib.pyplot as plt

# Machine Learning: Skalierung, Clustering, Dimensionsreduktion
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Dendrogramm für hierarchisches Clustering
from scipy.cluster.hierarchy import dendrogram, linkage

# Standardgröße für alle Diagramme
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Bereinigten Datensatz laden

Der Datensatz wurde in Notebook 01 bereinigt. Diese aufbereitete Datei wird hier direkt geladen.

In [ ]:
df = pd.read_csv(
    "../data/online_retail_data_cleaned.csv",
    parse_dates=["InvoiceDate"]
)

df.head()

## 3. Überblick über den Datensatz

Kurze Kontrolle, ob der Datensatz korrekt geladen wurde.

In [ ]:
# Anzahl Zeilen und Spalten
df.shape

In [ ]:
# Datentypen und fehlende Werte
df.info()

In [ ]:
# Statistische Kennzahlen der numerischen Spalten
df.describe()

## 4. Kundenbezogene Merkmale berechnen

Für das Clustering brauchen wir eine Tabelle mit **einer Zeile pro Kunde**.
Die Transaktionen werden daher auf Kundenebene aggregiert.

Berechnete Merkmale:
- `total_revenue`: Gesamtumsatz des Kunden
- `number_of_orders`: Anzahl eindeutiger Rechnungen
- `total_quantity`: Gesamtmenge aller bestellten Artikel
- `avg_order_value`: Durchschnittlicher Wert einer Transaktion
- `unique_products`: Anzahl unterschiedlicher Produkte
- `active_days`: Anzahl unterschiedlicher Tage mit Bestellungen

In [ ]:
customer_data = df.groupby("CustomerID").agg(
    total_revenue=("TotalPrice", "sum"),
    number_of_orders=("InvoiceNo", "nunique"),
    total_quantity=("Quantity", "sum"),
    avg_order_value=("TotalPrice", "mean"),
    unique_products=("StockCode", "nunique"),
    active_days=("InvoiceDate", lambda x: x.dt.date.nunique())
).reset_index()

customer_data.head()

## 5. Features für das Clustering auswählen

Für K-Means werden nur numerische Merkmale verwendet.
Die `CustomerID` ist lediglich eine Kennung und fließt nicht ins Modell ein.

In [ ]:
features = [
    "total_revenue",
    "number_of_orders",
    "total_quantity",
    "avg_order_value",
    "unique_products",
    "active_days"
]

X = customer_data[features]
X.head()

## 6. Log-Transformation und Skalierung

Online-Retail-Daten sind typischerweise stark rechtschief verteilt: Wenige Kund:innen
kaufen sehr viel, die Mehrheit kauft nur selten. Extreme Ausreißer würden K-Means
stark verzerren, da der Algorithmus empfindlich auf große Distanzen reagiert.

Deshalb wird zuerst eine **Log-Transformation** mit `np.log1p()` angewendet
(entspricht log(1 + x), funktioniert auch für Werte nahe 0).
Danach wird mit dem `StandardScaler` auf Mittelwert 0 und Standardabweichung 1 skaliert.

In [ ]:
# Log-Transformation: dämpft Ausreißer, erhält relative Unterschiede
X_log = np.log1p(X)

# Standardisierung: alle Merkmale werden vergleichbar gemacht
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

# Skalierte Daten als DataFrame zur Kontrolle
X_scaled_df = pd.DataFrame(X_scaled, columns=features)
X_scaled_df.head()

## 7. Elbow-Methode

Die Elbow-Methode testet K-Means für verschiedene Clusteranzahlen und misst jeweils
die Summe der quadratischen Abstände aller Punkte zu ihrem Clusterzentrum (SSE/Inertia).
Ein deutlicher Knick im Diagramm kann auf eine sinnvolle Clusteranzahl hinweisen.

In [ ]:
sse = []
k_values = range(1, 11)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    sse.append(kmeans.inertia_)

plt.plot(k_values, sse, marker="o")
plt.title("Elbow-Methode zur Bestimmung der Clusteranzahl")
plt.xlabel("Anzahl der Cluster k")
plt.ylabel("SSE / Inertia")
plt.xticks(k_values)
plt.tight_layout()
plt.savefig("../output/02_elbow_methode.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Silhouette Score

Der Silhouette Score misst für jeden Datenpunkt, wie gut er zum eigenen Cluster passt
und wie klar er von den anderen Clustern getrennt ist.
Werte nahe 1 bedeuten gute Trennung, Werte nahe 0 bedeuten Überlappung.

In [ ]:
silhouette_scores = []
k_values_silhouette = range(2, 11)

for k in k_values_silhouette:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

silhouette_df = pd.DataFrame({
    "k": list(k_values_silhouette),
    "silhouette_score": silhouette_scores
})

silhouette_df

In [ ]:
plt.plot(k_values_silhouette, silhouette_scores, marker="o")
plt.title("Silhouette Score für verschiedene Clusteranzahlen")
plt.xlabel("Anzahl der Cluster k")
plt.ylabel("Silhouette Score")
plt.xticks(k_values_silhouette)
plt.tight_layout()
plt.savefig("../output/02_silhouette_scores.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Wahl der Clusteranzahl: k = 3

Der höchste Silhouette Score liegt bei **k = 2**. Rein statistisch wäre das die
trennschärfste Lösung. Bei k = 2 entstehen jedoch nur zwei sehr grobe Gruppen
(Gelegenheitskäufer vs. aktive Kunden), die für eine differenzierte Kundensegmentierung
wenig Mehrwert bieten.

Wir entscheiden uns bewusst für **k = 3**, weil:
- Der Silhouette Score bei k = 3 (0.304) immer noch akzeptabel ist
- k = 3 die drei im Retail üblichen Kundensegmente abbildet:
  *Einmalkäufer*, *gelegentliche Kunden* und *Stammkunden*
- Die Elbow-Kurve zeigt, dass der größte Gewinn durch den Schritt von k=1 auf k=2
  und k=2 auf k=3 erzielt wird – danach flacht die Kurve stark ab
- k = 3 ist für eine Projektpräsentation deutlich besser interpretierbar

Diese Entscheidung ist ein Beispiel dafür, dass in der Praxis neben dem
mathematischen Optimum immer auch die **inhaltliche Sinnhaftigkeit** zählt.

In [ ]:
# Clusteranzahl bewusst auf 3 gesetzt (Begründung siehe oben)
best_k = 3
print(f"Gewählte Clusteranzahl: k = {best_k}")

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
customer_data["cluster"] = kmeans.fit_predict(X_scaled)

customer_data.head()

## 10. Clustergrößen

Wie viele Kund:innen gehören zu jedem Cluster?

In [ ]:
cluster_counts = customer_data["cluster"].value_counts().sort_index()
print(cluster_counts)

cluster_counts.plot(kind="bar")
plt.title("Anzahl der Kund:innen pro Cluster")
plt.xlabel("Cluster")
plt.ylabel("Anzahl Kund:innen")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../output/02_clustergroessen.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Cluster interpretieren

Die Durchschnittswerte pro Cluster zeigen, welche Art von Kund:innen
sich in den jeweiligen Gruppen befindet.

In [ ]:
# Durchschnittliche Merkmale pro Cluster
cluster_summary = customer_data.groupby("cluster")[features].mean().round(2)

# Nach Gesamtumsatz sortieren für bessere Lesbarkeit
cluster_summary.sort_values("total_revenue", ascending=False)

## 12. Interpretation der Cluster

Anhand der Durchschnittswerte lassen sich die drei Cluster inhaltlich beschreiben:

| Cluster | Beschreibung | Merkmale |
|---------|-------------|----------|
| Hoher Umsatz | **Stammkunden** | Viele Bestellungen, hoher Umsatz, viele Produkte, viele aktive Tage |
| Mittlerer Umsatz | **Gelegentliche Kunden** | Moderater Umsatz, einige Bestellungen |
| Niedriger Umsatz | **Einmalkäufer** | Wenige Bestellungen, geringer Umsatz, kaum aktive Tage |

*Die genaue Zuordnung der Cluster-Nummern hängt von den berechneten Werten ab
und kann beim Ausführen des Notebooks nachgeschlagen werden.*

## 13. PCA zur Visualisierung

Da das Clustering auf sechs Merkmalen basiert, lassen sich die Cluster nicht direkt
in einem 2D-Diagramm darstellen. PCA (Principal Component Analysis) reduziert die
Dimensionen auf zwei Hauptkomponenten, sodass eine Visualisierung möglich wird.

In [ ]:
# PCA auf 2 Komponenten reduzieren
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# PCA-Koordinaten zur Kundentabelle hinzufügen
customer_data["pca_1"] = X_pca[:, 0]
customer_data["pca_2"] = X_pca[:, 1]

customer_data.head()

In [ ]:
sc = plt.scatter(
    customer_data["pca_1"],
    customer_data["pca_2"],
    c=customer_data["cluster"],
    alpha=0.6,
    cmap="viridis"
)
plt.colorbar(sc, label="Cluster")
plt.title("Kundensegmente visualisiert mit PCA (k=3)")
plt.xlabel("PCA Komponente 1")
plt.ylabel("PCA Komponente 2")
plt.tight_layout()
plt.savefig("../output/02_pca_kundensegmente.png", dpi=150, bbox_inches="tight")
plt.show()

## 14. Erklärte Varianz der PCA

Der Anteil der erklärten Varianz zeigt, wie viel Information
durch die beiden PCA-Komponenten erhalten bleibt.

In [ ]:
explained_variance = pd.DataFrame({
    "component": ["PCA 1", "PCA 2"],
    "explained_variance_ratio": pca.explained_variance_ratio_
})

print(f"Erklärte Gesamtvarianz: {pca.explained_variance_ratio_.sum():.2%}")
explained_variance

## 15. Hierarchisches Clustering als Vergleich

Als zweite Methode wird hierarchisches Clustering verwendet.
Das Dendrogramm zeigt, wie Kund:innen schrittweise zu Gruppen zusammengeführt werden.
Für die Darstellung wird eine Stichprobe von 200 Kund:innen verwendet.

In [ ]:
# Stichprobe für das Dendrogramm (vollständige Daten wären zu unübersichtlich)
X_sample = X_scaled[:200]

linked = linkage(X_sample, method="ward")

plt.figure(figsize=(12, 6))
dendrogram(linked)
plt.title("Dendrogramm: Hierarchisches Clustering (Stichprobe, n=200)")
plt.xlabel("Kund:innen in der Stichprobe")
plt.ylabel("Distanz")
plt.tight_layout()
plt.savefig("../output/02_dendrogramm_hierarchisch.png", dpi=150, bbox_inches="tight")
plt.show()

## 16. Agglomeratives Clustering

Das hierarchische Clustering wird mit derselben Clusteranzahl (k=3) durchgeführt.
Die Kreuztabelle zeigt, wie stark die Zuordnungen beider Verfahren übereinstimmen.

In [ ]:
# Hierarchisches Clustering mit k=3
agglo = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
customer_data["cluster_hierarchical"] = agglo.fit_predict(X_scaled)

# Übereinstimmung der beiden Verfahren prüfen
pd.crosstab(
    customer_data["cluster"],
    customer_data["cluster_hierarchical"],
    rownames=["K-Means"],
    colnames=["Hierarchisch"]
)

## 17. Ergebnisse exportieren

In [ ]:
customer_data.to_csv(
    "../data/online_retail_customer_segments.csv",
    index=False
)

print("Datei gespeichert: ../data/online_retail_customer_segments.csv")

## 18. Fazit

In diesem Notebook wurden Kund:innen mithilfe von K-Means in **drei Segmente** eingeteilt:

- **Stammkunden**: hoher Umsatz, viele Bestellungen, viele aktive Tage
- **Gelegentliche Kunden**: moderater Umsatz, einige Bestellungen
- **Einmalkäufer**: sehr niedriger Umsatz, kaum Aktivität

Die Entscheidung für k = 3 wurde bewusst getroffen, da sie trotz leicht niedrigerem
Silhouette Score gegenüber k = 2 eine deutlich bessere inhaltliche Interpretation ermöglicht.

Im nächsten Notebook (03) wird diese Segmentierung durch ein Supervised-Classification-Modell
ergänzt, das gezielt vorhersagt, ob ein Kunde zu den umsatzstärksten gehört.